In [ ]:
import os
import sys
import scipy.io
from scipy.io import loadmat
import pandas as pd
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
sys.path.insert(0, '..')
from src.conf import settings
import json
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import shutil

In [ ]:
# Load the annotation file
#annotations = scipy.io.loadmat(r"C:\Users\Nagham\HRNet-Human-Pose-Estimation\data\hr-lspet\joints.mat")
# Load the MATLAB file
annotations = loadmat(settings.LABELS_PATH)
joints = annotations['joints']  # Shape will be (3, 14, N), where N is the number of images

# Check the type and shape of the joints array
print("Type of joints:", type(joints))
print("Shape of joints:", joints.shape)

In [ ]:
# Function to load .mat file
def load_mat(mat_path):
    data = scipy.io.loadmat(mat_path)
    return data

# Function to list all images in the directory
def list_images(image_dir):
    return [f for f in os.listdir(image_dir) if os.path.isfile(os.path.join(image_dir, f))]

# Function to create a dataframe with images and their annotations
def create_dataframe(image_dir, annotations):
    images = list_images(image_dir)
    records = []
    
    for i, image in enumerate(images):
        if i < annotations.shape[2]:  # Ensure we do not go out of bounds
            annotation = annotations[:, :, i]  # Assuming annotations is a 3D array
            record = {'image': image, 'annotations': annotation}
            records.append(record)
        else:
            print(f"No annotation found for image: {image}")

    df = pd.DataFrame(records)
    return df



In [ ]:
df = create_dataframe(settings.IMAGE_DIR, joints)
df.head()


In [ ]:
def load_json(file_path):
    if not os.path.exists(file_path):
        print(f"JSON file {file_path} does not exist.")
        return None
    
    try:
        with open(file_path, 'r') as file:
            data = json.load(file)
        return data
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON file {file_path}: {e}")
        return None
    except IOError as e:
        print(f"IOError reading JSON file {file_path}: {e}")
        return None

def save_json(data, file_path):
    with open(file_path, 'w') as file:
        json.dump(data, file, indent=4)

def is_point_in_bbox(point, bbox):
    x, y = point
    x_min, y_min, x_max, y_max = bbox
    return x_min <= x <= x_max and y_min <= y <= y_max

def find_json_file(start_letters, folder_path):
    for file_name in os.listdir(folder_path):
        if file_name.startswith(start_letters) and file_name.endswith('.json'):
            return os.path.join(folder_path, file_name)
    return None

def filter_bounding_boxes(image_name, json_folder_path, df):

    # Extract the first 7 letters of the image name
    start_letters = image_name[:7]
    
    # Find the corresponding JSON file
    json_file_path = find_json_file(start_letters, json_folder_path)
    
    if not json_file_path:
        print(f"No JSON file found starting with {start_letters} in {json_folder_path}")
        return
    

    # Load the JSON file
    json_data = load_json(json_file_path)

    if not json_file_path:
        print(f"No JSON file found starting with {start_letters} in {json_folder_path}")
        return # Exit the function if no JSON file is found
    
    # Load the JSON file
    json_data = load_json(json_file_path)
    
    if not json_data:
        return  # Exit the function if JSON data is empty
    
    # Extract keypoints for the image from the DataFrame
    keypoints = df[df['image'] == image_name]['annotations'].values[0]
    
    # Filter bounding boxes
    filtered_bboxes = []
    for entry in json_data:
        bbox = entry['bbox']
        if all(is_point_in_bbox(point, bbox) for point in keypoints[:,0:2]):
            filtered_bboxes.append(entry)
    
    # Update the JSON data with filtered bounding boxes
    json_data = filtered_bboxes
    
    # Save the updated JSON file
    save_json(json_data, json_file_path)
    print(f"Updated JSON file saved to {json_file_path}")
    return filtered_bboxes

def plot_bboxes_on_image(image_path, bboxes):
    # Load image
    image = Image.open(image_path)
    
    # Create figure and axes
    fig, ax = plt.subplots(1)
    ax.imshow(image)
    
    # Add bounding boxes to the image
    for bbox in bboxes:
        x_min, y_min, x_max, y_max = bbox['bbox']
        width = x_max - x_min
        height = y_max - y_min
        rect = patches.Rectangle((x_min, y_min), width, height, linewidth=1, edgecolor='r', facecolor='none')
        ax.add_patch(rect)
    
    plt.show()


In [ ]:
def process_all_images_in_folder(image_folder_path, json_folder_path, df):
    image_names_with_bboxes = []  # List to store image names with valid bounding boxes
    for file_name in os.listdir(image_folder_path):
        if file_name.endswith('.png') or file_name.endswith('.jpg'):  # Add other image extensions if necessary
            image_name = file_name
            image_path = os.path.join(image_folder_path, image_name)
            
            # Filter bounding boxes
            bboxes = filter_bounding_boxes(image_name, json_folder_path, df)
            
            # Plot bounding boxes on image if valid bboxes are found
            if not bboxes:
                print(f"No valid bounding boxes found for image {image_name}. Skipping.")
                image_names_with_bboxes.append(image_name)

    # Convert list of image names into a DataFrame
    image_names_df = pd.DataFrame(image_names_with_bboxes, columns=['image_name'])
    
    return image_names_df
            
# Example usage
json_folder_path = r'C:\Users\Nagham\Human-Pose-Estimation---AI-Development-Course\dataset\labels_yolo\test'
image_folder_path = r"C:\Users\Nagham\Human-Pose-Estimation---AI-Development-Course\dataset\images\test"

No_BBOX_image_names_df = process_all_images_in_folder(image_folder_path, json_folder_path, df)

In [ ]:
No_BBOX_image_names_df